# Séance 6 — Visualisation de données économiques

**Analyse des données — L3 Économie**

Un graphique n'est pas une illustration : c'est un **argument**. Il affirme quelque chose, et il peut mentir sans qu'aucune valeur soit fausse.

Toutes les figures de ce notebook sont justes. Plusieurs ne sont pas honnêtes.

> *Exécution → Tout exécuter* avant de démarrer.

## 0. Les données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL = ("https://raw.githubusercontent.com/StefaniaMarcassa/"
       "analyse_des_donnees/main/data/cm03-salaires.csv")

df = pd.read_csv(URL, sep=";", encoding="utf-8", decimal=",", thousands=" ")
df["SALAIRE_NET"] = df["SALAIRE_NET"].replace(999999, np.nan)
df = df[df["SALAIRE_NET"] != 0].copy()

s = df["SALAIRE_NET"].dropna()
print(len(df), "lignes |", len(s), "salaires observes")

## 1. Quatre questions, quatre graphiques

Le choix du graphique se fait sur **la question posée**, jamais sur l'esthétique.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 3.2))

# Comment se répartit une variable ?
ax[0].hist(s, bins=40)
ax[0].set_title("Une distribution")
ax[0].set_xlabel("salaire (€)")

# Qui gagne plus ?
g = df.groupby("SEXE")["SALAIRE_NET"].agg(["mean", "std", "count"])
g["err"] = g["std"] / np.sqrt(g["count"])
ax[1].bar(["Hommes", "Femmes"], g["mean"], yerr=1.96 * g["err"], capsize=4)
ax[1].set_title("Deux groupes")
ax[1].set_ylabel("salaire moyen (€)")

# Comment ça évolue ? (série illustrative, fabriquée)
rng = np.random.default_rng(4)
annees = np.arange(2000, 2025)
taux = 7.5 + np.cumsum(rng.normal(0, .45, len(annees))) + np.sin(np.arange(len(annees))/3) * .9
ax[2].plot(annees, taux)
ax[2].set_title("Une évolution")
ax[2].set_xlabel("année")

# Les deux vont-elles ensemble ?
ech = df.sample(700, random_state=1)
ax[3].scatter(ech["AGE"], ech["SALAIRE_NET"], s=5, alpha=.4)
ax[3].set_title("Une relation")
ax[3].set_xlabel("âge")

plt.tight_layout()
plt.show()

**Trois erreurs de choix fréquentes :**

| On voulait | On a tracé | Ce qui est perdu |
|---|---|---|
| Montrer une distribution | Une barre avec la moyenne | Toute la dispersion |
| Comparer deux groupes | Deux camemberts | L'œil compare mal les angles |
| Montrer une évolution | Des barres côte à côte | La continuité du temps |

## 2. L'axe tronqué

Mêmes données, deux messages.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))

for a, bas, titre in [(ax[0], 0, "Axe à zéro"), (ax[1], 2350, "Axe tronqué")]:
    a.bar(["Hommes", "Femmes"], g["mean"])
    a.set_ylim(bas, 2900)
    a.set_title(titre)
    a.set_ylabel("salaire moyen (€)")

plt.tight_layout()
plt.show()

print("ecart reel :", round(g["mean"][1] - g["mean"][2]), "euros, soit",
      round(100 * (g["mean"][1] - g["mean"][2]) / g["mean"][1], 1), "%")

L'écart vaut **336 euros**, soit 12 % — considérable, mais sans rapport avec ce que montre la figure de droite.

> **La règle.** Sur des **barres**, l'axe commence à zéro : la longueur doit être proportionnelle à la valeur. Sur une **courbe**, tronquer est légitime — on y lit une variation, pas une longueur.

## 3. Le double axe

Deux séries sans rapport. Observez leur corrélation avant de regarder les figures.

In [ ]:
rng = np.random.default_rng(11)
an = np.arange(2000, 2025)
taux_b  = 8 + np.cumsum(rng.normal(0, .35, len(an)))
indice  = 100 + 1.6 * np.arange(len(an)) + rng.normal(0, .4, len(an))

print("correlation reelle :", round(np.corrcoef(taux_b, indice)[0, 1], 2))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))

# deux axes : on choisit les echelles
a = ax[0]
a.plot(an, taux_b, color="black")
b = a.twinx()
b.plot(an, indice, color="red")
a.set_title("Deux axes : les courbes se suivent")

# un seul axe : memes donnees
ax[1].plot(an, taux_b, color="black", label="taux (%)")
ax[1].plot(an, indice, color="red", label="indice")
ax[1].set_title("Un seul axe : il n'y a rien à voir")
ax[1].legend()

plt.tight_layout()
plt.show()

Leur corrélation vaut **0,19**. À gauche elles semblent se suivre : il a suffi de choisir deux échelles.

> **Pourquoi c'est redoutable.** Avec deux axes libres, on fait coïncider presque n'importe quelles séries. Le lecteur voit une relation, et aucune valeur n'est fausse.

La parade : un seul axe, deux graphiques séparés, ou des séries en base 100.

## 4. La période choisie

In [ ]:
# chercher la fenetre de six ans ou la baisse est la plus forte
pires = [(taux[i+5] - taux[i], i) for i in range(len(annees) - 5)]
baisse, i = min(pires)
j = i + 5
print("fenetre la plus declinante :", annees[i], "-", annees[j], "| baisse de", round(-baisse, 1), "point")
print("sur l'ensemble :", round(taux[-1] - taux[0], 1), "point")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(annees, taux)
ax[0].set_title("2000 - 2024")
ax[1].plot(annees[i:j+1], taux[i:j+1], color="red", lw=2)
ax[1].set_title(f"{annees[i]} - {annees[j]}")
for a in ax:
    a.set_ylim(taux.min() - .4, taux.max() + .4)
    a.set_xlabel("année"); a.set_ylabel("taux (%)")
plt.tight_layout()
plt.show()

Même série. Sur vingt-cinq ans elle **monte** de 1,1 point. Sur les six années choisies, elle chute de 2 points. Les deux figures sont exactes.

> **Ce qu'il faut exiger.** Toute figure temporelle doit dire **pourquoi cette période**. Si le choix des bornes change la conclusion, c'est le choix qui fait la conclusion — pas les données.

## 5. Linéaire ou logarithmique

In [ ]:
annees_pib = np.arange(1965, 2025)
pib = 100 * 1.025 ** np.arange(len(annees_pib))   # exactement 2,5 % par an

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(annees_pib, pib)
ax[0].set_title("Échelle linéaire")
ax[1].plot(annees_pib, pib, color="green")
ax[1].set_yscale("log")
ax[1].set_title("Échelle logarithmique")
for a in ax: a.set_xlabel("année")
plt.tight_layout()
plt.show()

Une croissance de 2,5 % par an pendant soixante ans. À gauche, l'accélération paraît récente. À droite, la droite révèle la vérité : **le taux n'a jamais changé**.

> **Quand passer au logarithme.** Dès qu'on s'intéresse à des **taux de croissance** plutôt qu'à des niveaux : une pente constante y signifie un taux constant, deux pentes parallèles deux croissances identiques.

## 6. Une figure présentable

Les figures ci-dessus sont brutes. Une figure destinée à être lue par quelqu'un d'autre porte un **titre**, des **axes nommés avec leur unité**, et la **source** des données.

On part du défaut, puis on ajoute une ligne à la fois.


In [ ]:
# Ce que matplotlib donne par defaut
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(s, bins=40)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))

ax.hist(s, bins=40, color="#9AA0A0", edgecolor="white", lw=.4)

# un repere : la mediane
ax.axvline(s.median(), color="#E2162F", lw=1.6)
ax.text(s.median() + 150, ax.get_ylim()[1] * .88,
        f"médiane  {round(s.median())} €", color="#E2162F", fontsize=9)

ax.set_title("Distribution des salaires nets mensuels")
ax.set_xlabel("Salaire net mensuel (euros)")      # l'unite, toujours
ax.set_ylabel("Nombre de salariés")
ax.spines[["top", "right"]].set_visible(False)

fig.text(0.02, -0.02,
         f"Source : EEC 2024, salaires renseignés et non nuls (n = {len(s)}).",
         fontsize=8, color="#7A8280")
plt.show()


**Le test.** Si quelqu'un qui n'a pas lu votre texte peut dire *ce que la figure mesure*, *sur qui*, et *d'où viennent les chiffres*, elle est présentable. Quatre lignes de code suffisent.


## Quatre questions devant n'importe quel graphique

1. **Où commence l'axe vertical ?** S'il ne part pas de zéro sur des barres, l'écart est exagéré.
2. **Combien d'axes verticaux ?** Deux axes fabriquent des corrélations.
3. **Pourquoi cette période ?** Des bornes choisies produisent la tendance voulue.
4. **Combien d'observations derrière chaque point ?** Une moyenne sur sept personnes se dessine comme une moyenne sur sept mille.

Ces questions valent d'abord pour **vos propres figures**. Le but n'est pas de débusquer les tricheurs, mais de ne pas en devenir un par négligence — le cas de loin le plus fréquent.

---

Supports, données et corrigés : `stefaniamarcassa.github.io/analyse_des_donnees`